# Checkpoints and restart

A rolling checkpoint atomically stores recipe position, partial
operation state, device geometry, active topology, refinement history,
cell bounds, and active targets. Scratch contact buffers are rebuilt on
resume. Cross-revision compatibility is not guaranteed: archive the
generating code revision, settings, backend, and an independent geometry
export. Use a new output path when branching a cleanup study.

In [ ]:
# Path objects are accepted anywhere TANGLE expects a file path.
from pathlib import Path
import tangle

## Every `CheckpointSettings` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `case_id` | Identity used to reject an incompatible restart. | nonempty string |
| `path` | Rolling checkpoint output path. | filesystem path |
| `interval_iterations` | Sparse save cadence. | positive iteration count |
| `resume` | Loads a checkpoint before executing. | boolean |
| `resume_path` | Optional source path distinct from the new output path. | path or `None` |
| `resume_case_id` | Optional expected identity for the source checkpoint. | string or `None` |
| `fresh_formation_on_resume` | Keeps geometry/history but restarts recipe operations. | boolean |

In [ ]:
# case_id prevents accidentally resuming an unrelated recipe that used
# the same filesystem location.
checkpoint = tangle.CheckpointSettings(
    "twenty-ply-needled-v1",
    Path("output/twenty_ply.restart"),
    interval_iterations=500,
    resume=False,
)
fields = [
    "case_id", "path", "interval_iterations", "resume",
    "resume_path", "resume_case_id", "fresh_formation_on_resume",
]
{name: getattr(checkpoint, name) for name in fields}

Set `resume=True` to continue the same rolling file. To branch a saved
state into a new experiment, use a new `path` and point `resume_path`
at the old file. `resume_case_id` validates the source identity.
`fresh_formation_on_resume=True` retains geometry/history but starts the
supplied recipe from operation zero.

In [ ]:
# Branching reads the old checkpoint but writes future progress to a new
# rolling file, leaving the source restart untouched.
branch = checkpoint.replace(
    path=Path("output/alternate_cleanup.restart"),
    resume=True,
    resume_path=checkpoint.path,
    resume_case_id=checkpoint.case_id,
    fresh_formation_on_resume=True,
)

# Expensive execution is deliberately opt-in in this reference notebook.
RUN_RECIPE = False
if RUN_RECIPE:
    recipe = tangle.Recipe(tangle.Cell([1e-3, 1e-3, 1e-3]))
    result = recipe.run(tangle.RelaxationSettings(), checkpoint=branch)
    # The resumed, relaxed geometry is a new Assembly on the result.
    print(result.resumed, result.resumed_iteration, result.assembly.fiber_count)